# 04 · LGD Estimation

Combines the two components with the formula of the thesis (eq. 1, ING):

$$\widehat{LGD} = (1 - \widehat{P}(\text{cure})) \times (1 - \widehat{RR}) + \frac{\text{costs}}{EAD}$$

The dataset has no workout costs, so the cost term is 0. The **realised** LGD of a loan is 0 if it cured and
`1 − rec_rate` otherwise; loans without a known outcome (still in default, no loss figures) are left out of the
evaluation.

1. Choose the cure model (validation set) and the recovery model (repeated cross-validation from notebook 03)
2. Loan-level LGD accuracy on the test set (with bootstrap confidence intervals)
3. Portfolio LGD (EAD-weighted) – predicted vs realised
4. Every combination of cure × recovery model
5. Calibration and distribution plots

Run notebooks 01–03 first.

In [ ]:
# Reload the lgd package automatically when its files change (no kernel restart needed after edits).
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

from lgd.config import OUTPUT_DIR, PREDICTION_DIR
from lgd.data import load_model_dataset
from lgd.metrics import bootstrap_ci, format_ci, regression_metrics
from lgd.plots import save_and_show

dataset = load_model_dataset()
realised_lgd = pd.Series(np.where(dataset["cure"] == 1, 0.0, 1 - dataset["rec_rate"]), index=dataset.index)  # NaN = unknown
ead = dataset["EAD"]


def load_predictions(component: str, split: str) -> pd.DataFrame:
    return pd.read_csv(PREDICTION_DIR / f"{component}_{split}.csv", index_col="loan_id")


cure_valid, cure_test = load_predictions("cure", "valid"), load_predictions("cure", "test")
recovery_valid, recovery_test = load_predictions("recovery", "valid"), load_predictions("recovery", "test")

In [ ]:
def lgd_estimate(p_cure, recovery_rate):
    return (1 - p_cure) * (1 - np.clip(recovery_rate, 0, 1))

## 1. Model Selection

* Cure model with the lowest **Brier score** (the probability itself enters the formula, so it must be
  well calibrated, not only well ranked).
* Recovery model with the lowest **repeated cross-validation MSE** from notebook 03 (the validation set has only a
  few hundred recovery rates); if that file is missing, the lowest MSE on the non-cured validation loans.

In [ ]:
valid_ids = cure_valid.index
cure_brier = ((cure_valid.sub(dataset.loc[valid_ids, "cure"], axis=0)) ** 2).mean().sort_values()

non_cured_valid = valid_ids[(dataset.loc[valid_ids, "cure"] == 0) & dataset.loc[valid_ids, "rec_rate"].notna()]
recovery_mse = (recovery_valid.loc[non_cured_valid].sub(dataset.loc[non_cured_valid, "rec_rate"], axis=0) ** 2).mean().sort_values()

best_cure_model = cure_brier.index[0]
selection_file = OUTPUT_DIR / "recovery_model_selection_cv.csv"
if selection_file.exists():
    recovery_cv = pd.read_csv(selection_file, index_col=0)
    best_recovery_model = recovery_cv.index[0]
    recovery_reason = f"repeated CV MSE = {recovery_cv['CV MSE'].iloc[0]:.5f}"
else:
    recovery_cv, best_recovery_model = None, recovery_mse.index[0]
    recovery_reason = f"validation MSE = {recovery_mse.iloc[0]:.5f}"

print(f"Cure model:     {best_cure_model}  (validation Brier = {cure_brier.iloc[0]:.5f})")
print(f"Recovery model: {best_recovery_model}  ({recovery_reason})")
columns = {"Cure: validation Brier": cure_brier, "Recovery: validation MSE": recovery_mse}
if recovery_cv is not None:
    columns["Recovery: repeated CV MSE"] = recovery_cv["CV MSE"]
pd.concat(columns, axis=1)

## 2. Loan-Level LGD on the Test Set

In [ ]:
test_ids = cure_test.index[realised_lgd.loc[cure_test.index].notna()]   # loans with a known outcome
y_test = realised_lgd.loc[test_ids]
print(f"{len(test_ids)} of {len(cure_test)} test loans have a known realised LGD")

lgd_models = {
    "Selected models": lgd_estimate(cure_test.loc[test_ids, best_cure_model], recovery_test.loc[test_ids, best_recovery_model]),
    "Baseline (averages only)": lgd_estimate(cure_test.loc[test_ids, "Baseline"], recovery_test.loc[test_ids, "Baseline"]),
}

loan_level = {}
for name, prediction in lgd_models.items():
    metrics = regression_metrics(y_test, prediction)
    metrics["MAE 95% CI"] = format_ci(metrics["MAE"], bootstrap_ci(y_test, prediction, mean_absolute_error))
    loan_level[name] = metrics
pd.DataFrame(loan_level).T

## 3. Portfolio LGD (EAD-weighted)

`Portfolio LGD = Σ LGDᵢ · EADᵢ / Σ EADᵢ` over the test loans. The confidence interval resamples the loans.

In [ ]:
def portfolio_lgd(lgd_values, weights):
    return float(np.sum(lgd_values * weights) / np.sum(weights))


weights = ead.loc[test_ids].to_numpy()
predicted = lgd_models["Selected models"].to_numpy()
actual = y_test.to_numpy()

rng = np.random.default_rng(0)
bootstrap_gap = []
for idx in rng.integers(0, len(actual), size=(1000, len(actual))):
    bootstrap_gap.append(portfolio_lgd(predicted[idx], weights[idx]) - portfolio_lgd(actual[idx], weights[idx]))

portfolio = pd.Series({
    "Realised portfolio LGD": portfolio_lgd(actual, weights),
    "Predicted portfolio LGD": portfolio_lgd(predicted, weights),
    "Predicted − realised": portfolio_lgd(predicted, weights) - portfolio_lgd(actual, weights),
    "95% CI low": np.quantile(bootstrap_gap, 0.025),
    "95% CI high": np.quantile(bootstrap_gap, 0.975),
})
portfolio.round(5)

## 4. Every Cure × Recovery Combination (test MAE)

For information only: the choice was made on the validation set above, picking the best cell here would
use the test set for model selection.

In [ ]:
combination_mae = pd.DataFrame(
    {recovery_name: {cure_name: mean_absolute_error(y_test, lgd_estimate(cure_test.loc[test_ids, cure_name], recovery_test.loc[test_ids, recovery_name]))
                     for cure_name in cure_test.columns}
     for recovery_name in recovery_test.columns}
).rename_axis(index="cure model", columns="recovery model")
combination_mae.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r", axis=None)

## 5. Calibration and Distribution

In [ ]:
deciles = pd.qcut(predicted, 10, labels=False, duplicates="drop")
calibration = pd.DataFrame({"predicted": predicted, "realised": actual, "decile": deciles}).groupby("decile").mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(calibration["predicted"], calibration["realised"], "o-", label="Selected models")
axes[0].plot([0, 1], [0, 1], "k--", label="Perfect calibration")
axes[0].set(title="LGD Calibration by Decile (test set)", xlabel="Mean predicted LGD", ylabel="Mean realised LGD")
axes[0].legend()
axes[0].grid(True)

axes[1].hist(actual, bins=50, alpha=0.6, label="Realised LGD")
axes[1].hist(predicted, bins=50, alpha=0.6, label="Predicted LGD")
axes[1].set(title="Distribution of LGD (test set)", xlabel="LGD", ylabel="Loans")
axes[1].legend()
save_and_show(fig, "lgd_calibration_and_distribution.png")

> The realised LGD is exactly 0 for every cured loan, so its distribution has a spike at 0 that a
> probability-weighted prediction never reproduces. Judge the LGD model on calibration and portfolio
> accuracy rather than on loan-level R².

In [ ]:
lgd_output = pd.DataFrame({
    "p_cure": cure_test.loc[test_ids, best_cure_model],
    "recovery_rate": recovery_test.loc[test_ids, best_recovery_model],
    "lgd_predicted": lgd_models["Selected models"],
    "lgd_realised": y_test,
    "EAD": ead.loc[test_ids],
})
lgd_output.to_csv(OUTPUT_DIR / "lgd_test_predictions.csv")
lgd_output.describe().T